# Tutorial 12: Universal Graph Neural Networks for Quantum Circuit Design

## Why Graph ML?

Tutorial 8 trained a tabular DNN mapping design parameters → Hamiltonian targets. This works for a **fixed** qubit-claw-resonator-feedline topology. But add a second resonator? Remove the feedline? The model breaks.

The **Universal GNN** replaces tabular features with a **heterogeneous graph** of geometric embeddings. The key idea:

1. **Design parameters** → `build_layout()` → Shapely polygons
2. Each component → **static embedding** = `param_sum ∥ geometric_moments ∥ shape_tensor`
3. Connections → **typed physical edges** with coupling type, distances, overlap
4. Full layout → **virtual hub node** connecting to all components
5. **HeteroConv GNN** learns which design parameters affect which Hamiltonian targets

**All nodes predict all targets** during training. The GNN learns the physics through message passing — we never manually filter which node predicts what. At inference on a new topology, we use a simple readout map to know which predictions to extract from which node types.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.manifold import TSNE
from sklearn.metrics import r2_score
from torch_geometric.loader import DataLoader

from squadds.ml.universal.geometry.layout import build_layout
from squadds.ml.universal.geometry.viz import plot_layout, plot_component
from squadds.ml.universal.features.node_encoder import (
    compute_static_embedding, get_polygon_for_component,
    static_embedding_dim, DEFAULT_SHAPE_RESOLUTION,
)
from squadds.ml.universal.features.moments import compute_moments, moment_names
from squadds.ml.universal.features.edge_extractor import EdgeFeatureExtractor, edge_feature_dim
from squadds.ml.universal.graph.netlist import CircuitNetlist, ComponentSpec, EdgeSpec
from squadds.ml.universal.graph.builder import UniversalGraphBuilder
from squadds.ml.universal.graph.virtual_hub import _rasterize_in_bounds, spatial_edge_feature_dim
from squadds.ml.universal.model.gat_model import (
    UniversalGNN, NODE_TARGET_NAMES, INFERENCE_READOUT,
)
from squadds.ml.universal.trainer import UniversalTrainer

SHAPE_RES = DEFAULT_SHAPE_RESOLUTION
print(f"Shape resolution: {SHAPE_RES}x{SHAPE_RES} = {SHAPE_RES**2} dims")
print(f"Node embedding dim: {static_embedding_dim(SHAPE_RES)}")
print(f"Edge feature dim: {edge_feature_dim(SHAPE_RES)}")

seed = 42
torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)


---\n## 1. From Design Parameters to Physical Layout

In [ ]:
df = pd.read_parquet("data/training_data.parquet").drop_duplicates().reset_index(drop=True)
print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Design params: cross_length, cross_gap, claw_length, ground_spacing, coupling_length, total_length")
print(f"Targets: qubit_frequency_GHz, anharmonicity_MHz, cavity_frequency_GHz, kappa_kHz, g_MHz")
df.head(3)


In [ ]:
row = df.iloc[0]
lyt = build_layout(
    cross_length=row["cross_length"], cross_gap=row["cross_gap"],
    claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
    coupling_length=row["coupling_length"], total_length=row["total_length"],
)
fig = plot_layout(lyt)
plt.suptitle(f"Layout: cross_length={row['cross_length']}, claw_length={row['claw_length']}", y=1.01)
plt.show()


---
## 2. Static Embedding: Polygon → Fixed-Size Vector

Each component → deterministic embedding: `param_sum (1) ∥ moments (8) ∥ shape_tensor (R²)`

This is **universal**: same dimensionality regardless of component type, living in the same vector space.


In [ ]:
comp_names = ["qubit", "claw", "resonator", "feedline"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, name in enumerate(comp_names):
    polygon = get_polygon_for_component(lyt[name])
    params = lyt[name].get("params", {})
    embedding = compute_static_embedding(polygon, params=params, shape_resolution=SHAPE_RES)
    moments = compute_moments(polygon)
    
    print(f"=== {name.upper()} ===")
    print(f"  param_sum: {sum(params.values()) if params else 0:.1f}")
    for mn, mv in zip(moment_names(), moments):
        print(f"  {mn:20s}: {mv:12.2f}")
    print(f"  embedding norm: {np.linalg.norm(embedding):.2f}")
    print()
    
    shape_img = embedding[9:].reshape(SHAPE_RES, SHAPE_RES)
    axes[0, i].imshow(shape_img, cmap='viridis', interpolation='nearest')
    axes[0, i].set_title(f"{name.title()} Shape Tensor"); axes[0, i].axis('off')
    axes[1, i].barh(moment_names(), moments, color='steelblue')
    axes[1, i].set_title(f"{name.title()} Moments"); axes[1, i].tick_params(labelsize=7)

plt.suptitle("Static Embeddings: Shape Tensors & Moments", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


### Embedding Space from Dataset

In [ ]:
embs, labels = [], []
for _, r in df.head(200).iterrows():
    ly = build_layout(cross_length=r["cross_length"], cross_gap=r["cross_gap"],
                     claw_length=r["claw_length"], ground_spacing=r["ground_spacing"],
                     coupling_length=r["coupling_length"], total_length=r["total_length"])
    for cn in comp_names:
        embs.append(compute_static_embedding(get_polygon_for_component(ly[cn]),
                    params=ly[cn].get("params",{}), shape_resolution=SHAPE_RES))
        labels.append(cn.title())
X_2d = TSNE(perplexity=30, random_state=42).fit_transform(np.array(embs))
plt.figure(figsize=(10,7))
sns.scatterplot(x=X_2d[:,0], y=X_2d[:,1], hue=labels, palette="deep", s=60, alpha=0.8)
plt.title("t-SNE of Component Embeddings (200 SQuADDS samples)"); plt.grid(alpha=0.3); plt.show()


---
## 3. Heterogeneous Graph Assembly

| Node Type | Description |
|---|---|
| `component` | Static embedding per component (param_sum + moments + shape) |
| `virtual` | Full layout union embedding + layer stack |

| Edge Type | Description |
|---|---|
| `physical` | component ↔ component: coupling_type + center dist + overlap geometry + overlap shape |
| `spatial_in` | component → virtual: rel_center + area_frac + perim_frac + masked shape |
| `spatial_out` | virtual → component: (same as spatial_in) |

**Key**: ALL nodes predict ALL 5 targets. The GNN learns the correlations.


In [ ]:
netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator", component_type="RouteMeander"),
        ComponentSpec(name="feedline", component_type="CoupledLineTee"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator", coupling_type="galvanic"),
        EdgeSpec(src="resonator", dst="feedline", coupling_type="capacitive"),
    ],
)

builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")
data = builder.build(lyt, netlist, global_features={"dielectric_constant": 11.45})

print("=== HeteroData Graph ===")
print(data)
print()
print("Inference readout map (metadata only — NOT used in training):")
for name, ctype in zip(data['component'].component_name, data['component'].component_type):
    readout = INFERENCE_READOUT.get(ctype, [])
    print(f"  {name:12s} ({ctype:16s}): reads {readout}")


In [ ]:
# Edge overlap shape tensors
edge_extractor = EdgeFeatureExtractor(shape_resolution=SHAPE_RES)
edge_pairs = [("qubit","claw","capacitive"),("claw","resonator","galvanic"),("resonator","feedline","capacitive")]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (src, dst, ctype) in enumerate(edge_pairs):
    feat = edge_extractor.extract(
        get_polygon_for_component(lyt[src]),
        get_polygon_for_component(lyt[dst]), coupling_type=ctype)
    print(f"{src}->{dst} ({ctype}): coupling={feat[:3]}, dx={feat[3]:.1f}, dy={feat[4]:.1f}, overlap_area={feat[5]:.1f}")
    axes[i].imshow(feat[8:].reshape(SHAPE_RES, SHAPE_RES), cmap='hot', interpolation='nearest')
    axes[i].set_title(f"{src} <-> {dst}"); axes[i].axis('off')
plt.suptitle("Physical Edge: Overlap Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Hub masked shape tensors
from shapely.ops import unary_union
cpols = [get_polygon_for_component(lyt[n]) for n in comp_names]
lu = unary_union(cpols)
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i, (name, poly) in enumerate(zip(comp_names, cpols)):
    axes[i].imshow(_rasterize_in_bounds(poly, lu.bounds, SHAPE_RES), cmap='Blues', interpolation='nearest')
    axes[i].set_title(f"Hub->{name.title()}"); axes[i].axis('off')
    print(f"Hub->{name:12s}: area_frac={poly.area/lu.area:.4f}, perim_frac={poly.length/lu.length:.4f}")
axes[4].imshow(_rasterize_in_bounds(lu, lu.bounds, SHAPE_RES), cmap='Greens', interpolation='nearest')
axes[4].set_title("Hub (Full)"); axes[4].axis('off')
plt.suptitle("Spatial Edges: Masked Shape Tensors", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 4. Training

Each row → HeteroData graph. **ALL nodes get ALL 5 targets** — the GNN learns which parameters affect which targets through message passing.


In [ ]:
N_SAMPLES = 5000  # Increase for production
df_sub = df.head(N_SAMPLES)

# Target scaling for better convergence
TARGET_SCALES = {
    "qubit_frequency_GHz": 1.0,
    "anharmonicity_MHz": 100.0,
    "cavity_frequency_GHz": 1.0,
    "kappa_kHz": 100.0,
    "g_MHz": 100.0,
}

graph_dataset = []
builder = UniversalGraphBuilder(shape_resolution=SHAPE_RES, cache_dir="graph_cache")

print(f"Building {N_SAMPLES} graphs...")
for idx, row in df_sub.iterrows():
    lyt_i = build_layout(
        cross_length=row["cross_length"], cross_gap=row["cross_gap"],
        claw_length=row["claw_length"], ground_spacing=row["ground_spacing"],
        coupling_length=row["coupling_length"], total_length=row["total_length"],
    )
    data_i = builder.build(lyt_i, netlist, global_features={"dielectric_constant": 11.45})
    
    # ALL nodes get ALL 5 targets (no filtering!)
    y = data_i["component"].y.clone()
    for i in range(y.size(0)):
        y[i, 0] = row["qubit_frequency_GHz"] / TARGET_SCALES["qubit_frequency_GHz"]
        y[i, 1] = row["anharmonicity_MHz"] / TARGET_SCALES["anharmonicity_MHz"]
        y[i, 2] = row["cavity_frequency_GHz"] / TARGET_SCALES["cavity_frequency_GHz"]
        y[i, 3] = row["kappa_kHz"] / TARGET_SCALES["kappa_kHz"]
        y[i, 4] = row["g_MHz"] / TARGET_SCALES["g_MHz"]
    data_i["component"].y = y
    graph_dataset.append(data_i)
    if (idx + 1) % 1000 == 0:
        print(f"  {idx+1}/{N_SAMPLES}")

print(f"Done: {len(graph_dataset)} graphs")
print(f"Sample targets (all nodes identical for standard topology):")
print(f"  {graph_dataset[0]['component'].y[0]}")


In [ ]:
split = int(0.85 * len(graph_dataset))
train_loader = DataLoader(graph_dataset[:split], batch_size=32, shuffle=True)
val_loader = DataLoader(graph_dataset[split:], batch_size=32)

s = graph_dataset[0]
model = UniversalGNN(
    comp_dim=s["component"].x.size(1),
    virt_dim=s["virtual"].x.size(1),
    phys_edge_dim=s["component","physical","component"].edge_attr.size(1),
    spat_edge_dim=s["component","spatial_in","virtual"].edge_attr.size(1),
    hidden_dim=128, num_layers=3, num_heads=4, edge_hidden=32,
)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer = UniversalTrainer(model, learning_rate=1e-3, checkpoint_dir="checkpoints")
history = trainer.train_loop(train_loader, val_loader, epochs=500, patience=50)
trainer.load_checkpoint("best_model.pt")


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train", lw=2)
plt.plot(history["val_loss"], label="Val", lw=2)
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.title("Training Curve")
plt.legend(); plt.grid(alpha=0.3); plt.show()


---\n## 5. Parity Plots

In [ ]:
model.eval()
all_yt, all_yp = [], []
with torch.no_grad():
    for batch in val_loader:
        out = model(batch)
        all_yt.append(batch["component"].y)
        all_yp.append(out["node_preds"])

yt = torch.cat(all_yt).numpy()
yp = torch.cat(all_yp).numpy()

target_info = [
    ("Qubit Freq (GHz)", 0, 1.0),
    ("Anharmonicity (MHz)", 1, 100.0),
    ("Cavity Freq (GHz)", 2, 1.0),
    ("Kappa (kHz)", 3, 100.0),
    ("g (MHz)", 4, 100.0),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, (name, idx, scale) in enumerate(target_info):
    t = yt[:, idx] * scale
    p = yp[:, idx] * scale
    axes[i].scatter(t, p, alpha=0.3, s=8, color='crimson')
    lo, hi = min(t.min(), p.min()), max(t.max(), p.max())
    if lo != hi:
        axes[i].plot([lo, hi], [lo, hi], 'k--', lw=2)
        r2 = r2_score(t, p)
        axes[i].text(0.05, 0.9, f"R2 = {r2:.3f}", transform=axes[i].transAxes, fontsize=12,
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    axes[i].set(title=name, xlabel="True", ylabel="Predicted"); axes[i].grid(alpha=0.3)
axes[-1].axis('off')
plt.suptitle("Parity Plots: All Targets", fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6. Scale Invariance: The Holy Grail

The model was trained on **qubit-claw-resonator-feedline** graphs with all 5 targets on all nodes. At inference, we use the **INFERENCE_READOUT** map to extract the relevant predictions from each node type.

### Case 1: Qubit-Claw Only


In [ ]:
test_row = df.iloc[-1]
lyt_test = build_layout(
    cross_length=test_row["cross_length"], cross_gap=test_row["cross_gap"],
    claw_length=test_row["claw_length"], ground_spacing=test_row["ground_spacing"],
    coupling_length=test_row["coupling_length"], total_length=test_row["total_length"],
)

reduced_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
    ],
    edges=[EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive")],
)

data_r = builder.build(lyt_test, reduced_netlist, global_features={"dielectric_constant": 11.45})
model.eval()
with torch.no_grad():
    out_r = model(data_r)

scales = [1.0, 100.0, 1.0, 100.0, 100.0]
print("=== Case 1: Qubit-Claw Only ===")
print(f"Graph: {data_r['component'].x.size(0)} components, "
      f"{data_r['component','physical','component'].edge_index.size(1)} physical edges\n")

for i, (name, ctype) in enumerate(zip(data_r['component'].component_name, data_r['component'].component_type)):
    readout = INFERENCE_READOUT.get(ctype, [])
    preds = out_r['node_preds'][i]
    vals = {NODE_TARGET_NAMES[j]: preds[j].item()*scales[j] for j in range(5) if NODE_TARGET_NAMES[j] in readout}
    print(f"  {name:12s} ({ctype:16s}): {vals}")

print(f"\nGround truth: qubit_freq={test_row['qubit_frequency_GHz']:.3f}, "
      f"anharmonicity={test_row['anharmonicity_MHz']:.2f}, g={test_row['g_MHz']:.2f}")
print("Note: no cavity_freq/kappa in readout since no RouteMeander present.")

fig, ax = plt.subplots(figsize=(6, 6))
plot_component(lyt_test["qubit"], "qubit", ax=ax, show_etch=False)
plot_component(lyt_test["claw"], "claw", ax=ax, show_etch=False)
ax.autoscale_view(); ax.set_title("Case 1: Qubit-Claw Only", fontweight='bold'); plt.show()


### Case 2: Extended Topology (6 components)

Add a second feedline and resonator. The model automatically provides `cavity_freq` and `kappa` readouts for `resonator2`.


In [ ]:
lyt_ext = dict(lyt_test)
lyt_ext["feedline1"] = lyt_test["feedline"]
lyt_ext["resonator1"] = lyt_test["resonator"]
lyt_ext["feedline2"] = lyt_test["feedline"]
lyt_ext["resonator2"] = lyt_test["resonator"]
lyt_ext["design_params"] = lyt_test.get("design_params", {})

ext_netlist = CircuitNetlist(
    components=[
        ComponentSpec(name="qubit", component_type="TransmonCross"),
        ComponentSpec(name="claw", component_type="Claw"),
        ComponentSpec(name="resonator1", component_type="RouteMeander"),
        ComponentSpec(name="feedline1", component_type="CoupledLineTee"),
        ComponentSpec(name="feedline2", component_type="CoupledLineTee"),
        ComponentSpec(name="resonator2", component_type="RouteMeander"),
    ],
    edges=[
        EdgeSpec(src="qubit", dst="claw", coupling_type="capacitive"),
        EdgeSpec(src="claw", dst="resonator1", coupling_type="galvanic"),
        EdgeSpec(src="resonator1", dst="feedline1", coupling_type="capacitive"),
        EdgeSpec(src="feedline1", dst="feedline2", coupling_type="galvanic"),
        EdgeSpec(src="feedline2", dst="resonator2", coupling_type="capacitive"),
    ],
)

data_ext = builder.build(lyt_ext, ext_netlist, global_features={"dielectric_constant": 11.45})
with torch.no_grad():
    out_ext = model(data_ext)

print("=== Case 2: Extended Topology (6 components) ===")
print(f"Graph: {data_ext['component'].x.size(0)} components, "
      f"{data_ext['component','physical','component'].edge_index.size(1)} physical edges\n")

for i, (name, ctype) in enumerate(zip(data_ext['component'].component_name, data_ext['component'].component_type)):
    readout = INFERENCE_READOUT.get(ctype, [])
    preds = out_ext['node_preds'][i]
    vals = {NODE_TARGET_NAMES[j]: f"{preds[j].item()*scales[j]:.3f}" for j in range(5) if NODE_TARGET_NAMES[j] in readout}
    if vals:
        print(f"  {name:14s} ({ctype:16s}): {vals}")
    else:
        print(f"  {name:14s} ({ctype:16s}): (passive)")

print("\nresonator2 automatically gets cavity_freq and kappa predictions!")
print("The graph structure tells the model what to predict for each node type.")


---
## Summary

| | Tutorial 8 (Tabular DNN) | Tutorial 12 (Universal GNN) |
|---|---|---|
| Input | Fixed-size parameter vector | Heterogeneous graph of geometric embeddings |
| Feature space | Raw numbers | Shape tensors + moments + graph structure |
| Training | Each output mapped manually | All nodes learn all targets; GNN learns correlations |
| Inference on new topology | Impossible | Seamless — just build the new graph |
| Target readout | Single model output | Per-component-type readout map (scalable) |
